# Trabalho Final - Parte 2 - Parcial


# Estudo e Conceitos


## Processos em Python

### GIL do Python

O GIL (Global Interpreter Lock) é um bloqueio global que garante que apenas uma thread de Python bytecode esteja executando por vez dentro de um processo.

Isso significa que, mesmo criando várias threads, apenas uma thread consegue rodar instruções Python de cada vez.

Ou seja, por padrão threads em Python não aceleram tarefas CPU-bound.

### Multithreading e Multiprocessing

É possível criar múltiplas threads em Python através do módulo `threading` dentro de um mesmo processo Python, porém segue válida a limitação de executar apenas uma thread por vez dentro do mesmo processador, ou seja, as threads realizam chaveamento entre si em um processador, sem de fato executar em paralelo.

```python
import threading
import time

def tarefa(nome):
    for i in range(3):
        print(f"Tarefa {nome}: {i}")
        time.sleep(1)

# Criando duas threads
t1 = threading.Thread(target=tarefa, args=("A",))
t2 = threading.Thread(target=tarefa, args=("B",))

t1.start()
t2.start()

t1.join()
t2.join()
```


Com o objetivo de utilizar **múltiplos núcleos de CPU**, Python possui o módulo `multiprocessing`. Esse módulo cria **processos independentes** cada um com seu próprio interpretador e sua própria memória, evitando o GIL. Como resultado várias tarefas podem efetivamente executar em paralelo.

```python
from multiprocessing import Process
import math

def calcular():
    print(sum(i*i for i in range(10_000_000)))

processos = [Process(target=calcular) for _ in range(4)]

for p in processos:
    p.start()
for p in processos:
    p.join()
```

## Identificando IDs de núcleos do sistema

Um comando bastante comum para identificar os IDs dos núcleos de CPU disponíveis é o `lscpu`. Para visualizar apenas o ID dos núcleos pode-se utilziar `lscpu -p=CPU`. Alternativamente, `lscpu --extended` mostra, além dos IDs, mais detalhes sobre os núcleos, tal como número do socket de CPU, tamanho de memória cache e frequências de operação dos núcleos.

In [ ]:
!lscpu -p=CPU

In [ ]:
!lscpu --extended

## Identificando IDs de núcleos do sistema (com Python)

Outra forma é verificar com o módulo `psutil`, utilziando Python puro:

```python
import psutil
psutil.cpu_count(logical=True)   # Número de CPUs lógicas
psutil.cpu_count(logical=False)  # Número de núcleos físicos
```

In [ ]:
# Verificando IDs dos núcleos do sistema com Python puro (módulo psutil)

import psutil
print("Número de núcleos lógicos: " + str(psutil.cpu_count(logical=True)))
print("Número de núcleos físicos: " + str(psutil.cpu_count(logical=False)))

print("IDs lógicos disponíveis: " + str(list(range(psutil.cpu_count(logical=True)))))

## Afinidade de CPU (CPU affinity)

Por padrão o kernel do Linux é livre para mover um processo entre quaisquer núcleos de CPU disponíveis no sistema. No Ubuntu (Linux), entretanto, é possível controlar em qual CPU cada processo da aplicação Python irá executar com o uso da **afinidade de CPU**. 

Cada processo no Linux pode ter uma **máscara de afinidade** indicando quais núcleos ele está autorizado a executar.

### Comando os.sched_setaffinity()

Em Python, no Linux, a afinidade pode ser definidade utilizando `os.sched_setaffinity`. A chamada de sistema é utilizada diretamente no código:

```python
import os
import multiprocessing

def tarefa(cpu_id):
    # Define afinidade para o processo atual
    pid = os.getpid()
    os.sched_setaffinity(pid, {cpu_id})
    print(f"Processo {pid} rodando no CPU {cpu_id}")
    while True:
        pass  # loop infinito para ocupar CPU

if __name__ == "__main__":
    processos = []
    for cpu in range(2):  # usar apenas CPU 0 e CPU 1
        p = multiprocessing.Process(target=tarefa, args=(cpu,))
        processos.append(p)
        p.start()

    for p in processos:
        p.join()
```


In [ ]:
# Utilizando os.sched_setaffinity (Linux-only)
# Execução de processos em núcleos definidos

import os
import multiprocessing

def tarefa(cpu_id):
    # Define afinidade para o processo atual
    pid = os.getpid()
    os.sched_setaffinity(pid, {cpu_id})
    print(f"Processo {pid} rodando no CPU {cpu_id}")
    while True:
        pass  # loop infinito para ocupar CPU

if __name__ == "__main__":
    processos = []
    for cpu in range(2):  # usar apenas CPU 0 e CPU 1
        p = multiprocessing.Process(target=tarefa, args=(cpu,))
        processos.append(p)
        p.start()

    for p in processos:
        p.join()

### Utilizando `psutil` (multiplataforma)

O módulo `psutil`, diferentemente do comando `os.sched_setaffinity()` oferece portabilidade por ser multiplataforma, isto é, não dependente do Linux.

```python
import os
import psutil
import multiprocessing

def tarefa(cpu_id):
    pid = os.getpid()
    p = psutil.Process(pid)
    p.cpu_affinity([cpu_id])  # define a afinidade
    print(f"Processo {pid} rodando no CPU {cpu_id}")
    while True:
        pass

if __name__ == "__main__":
    processos = []
    for cpu in [0, 1, 2, 3]:  # escolher CPUs manualmente
        p = multiprocessing.Process(target=tarefa, args=(cpu,))
        processos.append(p)
        p.start()

    for p in processos:
        p.join()
```

Pode ser necessário instalar o `psutil` no sistema: 

```bash
sudo apt install python3-psutil
```

In [ ]:
# Utilizando psutil

cpu_ids = list(range(psutil.cpu_count(logical=True)))

import os
import psutil
import multiprocessing

def tarefa(cpu_id):
    pid = os.getpid()
    p = psutil.Process(pid)
    p.cpu_affinity([cpu_id])  # define a afinidade
    print(f"Processo {pid} rodando no CPU {cpu_id}")
    while True:
        pass

if __name__ == "__main__":
    processos = []
    #for cpu in [0, 1, 2, 3]:  # escolher núcleos manualmente
    for cpu in cpu_ids:  
        p = multiprocessing.Process(target=tarefa, args=(cpu,))
        processos.append(p)
        p.start()

    for p in processos:
        p.join()

## CPU governor

O **CPU governor** é uma política de escalonamento de frequência que o kernel do Linux aplica sobre os núcleos do processador. Ele é apenas uma camada de política, ou seja, um conjunto de regras de como usar a CPU. 

O governor decide **em qual frequência a CPU deve operar em um dado momento**, de acordo com critérios como:

- **Carga de trabalho** (se o sistema está ocioso ou ocupado).
- **Economia de energia** (modo portátil, servidor de baixo consumo).
- **Desempenho máximo** (quando a prioridade é velocidade).


### Principais tipos de governors

Dependendo do driver e da CPU, diferentes governors ficam disponíveis. Os mais comuns são:

**1. performance**
- Mantém a CPU sempre na frequência máxima.
- Vantagem: maior desempenho, útil para cargas pesadas e baixa latência.
- Desvantagem: maior consumo de energia e calor.
- Exemplo: ideal para servidores de banco de dados ou aplicações críticas.

**2. powersave**
- Mantém a CPU sempre na frequência mínima.
- Vantagem: reduz o consumo de energia.
- Desvantagem: desempenho limitado.
- Exemplo: bom para servidores de background ou quando o foco é economia.

**3. ondemand**
- Ajusta dinamicamente a frequência de acordo com a carga.
- Se a carga aumenta, sobe para máxima; se cai, reduz para mínima.
- Era muito usado em kernels antigos.

**4. conservative**
- Similar ao ondemand, mas aumenta/diminui a frequência de forma mais gradual.
- Útil para economizar energia sem oscilações bruscas.

**5. schedutil (moderno)**
- Integra o ajuste de frequência diretamente com o escalonador do kernel.
- Usa informações de agendamento de processos para decidir a frequência.
- É o padrão em muitas distros Linux recentes.


### Descobrindo os governos disponíveis

Comando:
```bash
cat /sys/devices/system/cpu/cpu0/cpufreq/scaling_available_governors
```

Saída:
```bash
performance powersave
```


In [3]:
# Verificando governors disponíveis
!cat /sys/devices/system/cpu/cpu0/cpufreq/scaling_available_governors

performance powersave


### Verificando o governor de cada núcleo de CPU


Comando:
```bash
cat /sys/devices/system/cpu/cpu*/cpufreq/scaling_governor
```

Saída:
```bash
powersave
powersave
powersave
powersave
powersave
powersave
powersave
powersave
```

Obs.: Considerando que há 8 núcleos disponíveis, isto é, CPU0 a CPU7

In [4]:
%%bash
# Governor de todas as CPUs

for cpu in /sys/devices/system/cpu/cpu*/cpufreq/scaling_governor; do
    num=$(echo $cpu | grep -oP 'cpu\K[0-9]+')
    echo "CPU $num: $(cat $cpu)"
done

CPU 0: performance
CPU 10: performance
CPU 11: performance
CPU 12: performance
CPU 13: performance
CPU 14: performance
CPU 15: performance
CPU 16: performance
CPU 17: performance
CPU 18: performance
CPU 19: performance
CPU 1: performance
CPU 20: performance
CPU 21: performance
CPU 22: performance
CPU 23: performance
CPU 24: performance
CPU 25: performance
CPU 26: performance
CPU 27: performance
CPU 2: performance
CPU 3: performance
CPU 4: performance
CPU 5: performance
CPU 6: performance
CPU 7: performance
CPU 8: performance
CPU 9: performance


### Alterar Governor

Alteração:
```bash
for cpu in /sys/devices/system/cpu/cpu*/cpufreq/scaling_governor; do
    echo performance | sudo tee $cpu
done
```

Verificação:
```bash
cat /sys/devices/system/cpu/cpu*/cpufreq/scaling_governor
```

In [19]:
# Alterando o governor: Código em Python para receber senha de sudo e executar alterações

import getpass
import os

# Pede a senha de sudo sem exibir na tela
password = getpass.getpass("Sudo password: ")

# Comando em bash que altera o governor para "performance"
bash_command = """
for cpu in /sys/devices/system/cpu/cpu*/cpufreq/scaling_governor; do
    echo performance | sudo tee $cpu
done
"""

# Monta o comando com sudo -S (lê senha do stdin)
command = f'echo "{password}" | sudo -S bash -c \'{bash_command}\''
os.system(command)

Sudo password: ········


[sudo] senha para jgross: 

performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance


0

In [20]:
%%bash
# Verificação final do governor, após modificação
cat /sys/devices/system/cpu/cpu*/cpufreq/scaling_governor

performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance
performance


## Verificando o driver da CPU

O **driver** da CPU é diferente do governor. Quem "fala" com o hardware é o driver, como `intel_pstate` ou `acpi-cpufreq`.

Comando:
```bash
cat /sys/devices/system/cpu/cpu*/cpufreq/scaling_driver
```

Saída:
```bash
intel_pstate
intel_pstate
intel_pstate
intel_pstate
intel_pstate
intel_pstate
intel_pstate
intel_pstate
```

Obs.: Considerando que há 8 núcleos disponíveis, isto é, CPU0 a CPU7

In [5]:
%%bash
# Driver de cada CPU

for cpu in /sys/devices/system/cpu/cpu*/cpufreq/scaling_driver; do
    num=$(echo $cpu | grep -oP 'cpu\K[0-9]+')
    echo "CPU $num: $(cat $cpu)"
done

CPU 0: intel_pstate
CPU 10: intel_pstate
CPU 11: intel_pstate
CPU 12: intel_pstate
CPU 13: intel_pstate
CPU 14: intel_pstate
CPU 15: intel_pstate
CPU 16: intel_pstate
CPU 17: intel_pstate
CPU 18: intel_pstate
CPU 19: intel_pstate
CPU 1: intel_pstate
CPU 20: intel_pstate
CPU 21: intel_pstate
CPU 22: intel_pstate
CPU 23: intel_pstate
CPU 24: intel_pstate
CPU 25: intel_pstate
CPU 26: intel_pstate
CPU 27: intel_pstate
CPU 2: intel_pstate
CPU 3: intel_pstate
CPU 4: intel_pstate
CPU 5: intel_pstate
CPU 6: intel_pstate
CPU 7: intel_pstate
CPU 8: intel_pstate
CPU 9: intel_pstate


## Verificando a frequência de operação dos núcleos

Comando:
```bash
cat /sys/devices/system/cpu/cpu*/cpufreq/scaling_driver
```

Saída:
```bash
intel_pstate
intel_pstate
intel_pstate
intel_pstate
intel_pstate
intel_pstate
intel_pstate
intel_pstate
```

Obs.: Considerando que há 8 núcleos disponíveis, isto é, CPU0 a CPU7

## Detalhamento da Arquitetura

### **CPU**
  - Modelo Intel(R) Core(TM) i7-14700T
  - Núcleos físicos: 20
    - p-cores: 8 (2 threads por núcleo)
      - Freq. turbo max: 5 GHz
      - Freq. base: 1.3 GHz
    - e-cores: 12 (1 thread por núcleo)
      - Freq. turbo max: 3.7 GHz
      - Freq. base: 900 MHz
  - Núcleos lógicos: 8 x 2 + 12 = 28  

In [13]:
!lscpu | grep NUMA

Nó(s) de NUMA:                           1
CPU(s) de nó0 NUMA:                      0-27


In [1]:
!lscpu --extended

CPU NODE SOCKET CORE L1d:L1i:L2:L3 ONLINE    MAXMHZ   MINMHZ       MHZ
  0    0      0    0 0:0:0:0          sim 5000,0000 800,0000  801,9680
  1    0      0    0 0:0:0:0          sim 5000,0000 800,0000 1102,1860
  2    0      0    1 4:4:1:0          sim 5000,0000 800,0000 1988,5430
  3    0      0    1 4:4:1:0          sim 5000,0000 800,0000  800,0000
  4    0      0    2 8:8:2:0          sim 5000,0000 800,0000 1531,1930
  5    0      0    2 8:8:2:0          sim 5000,0000 800,0000  800,0000
  6    0      0    3 12:12:3:0        sim 5000,0000 800,0000  824,6150
  7    0      0    3 12:12:3:0        sim 5000,0000 800,0000 1122,9170
  8    0      0    4 16:16:4:0        sim 5200,0000 800,0000 1803,1420
  9    0      0    4 16:16:4:0        sim 5200,0000 800,0000 1977,4880
 10    0      0    5 20:20:5:0        sim 5200,0000 800,0000 1507,0560
 11    0      0    5 20:20:5:0        sim 5200,0000 800,0000  800,0000
 12    0      0    6 24:24:6:0        sim 5000,0000 800,0000  800,0000
 13   

In [ ]:
# All p-core enabled (8 total) -> Hyper-Threading enabled (2 threads each)
# All e-cores enabled (12 total) -> 1 thread each
# Total logical cores: 28

CPU NODE SOCKET CORE L1d:L1i:L2:L3 ONLINE    MAXMHZ   MINMHZ       MHZ
  0    0      0    0 0:0:0:0          sim 5000,0000 800,0000  801,9680
  1    0      0    0 0:0:0:0          sim 5000,0000 800,0000 1102,1860
  2    0      0    1 4:4:1:0          sim 5000,0000 800,0000 1988,5430
  3    0      0    1 4:4:1:0          sim 5000,0000 800,0000  800,0000
  4    0      0    2 8:8:2:0          sim 5000,0000 800,0000 1531,1930
  5    0      0    2 8:8:2:0          sim 5000,0000 800,0000  800,0000
  6    0      0    3 12:12:3:0        sim 5000,0000 800,0000  824,6150
  7    0      0    3 12:12:3:0        sim 5000,0000 800,0000 1122,9170
  8    0      0    4 16:16:4:0        sim 5200,0000 800,0000 1803,1420
  9    0      0    4 16:16:4:0        sim 5200,0000 800,0000 1977,4880
 10    0      0    5 20:20:5:0        sim 5200,0000 800,0000 1507,0560
 11    0      0    5 20:20:5:0        sim 5200,0000 800,0000  800,0000
 12    0      0    6 24:24:6:0        sim 5000,0000 800,0000  800,0000
 13    0      0    6 24:24:6:0        sim 5000,0000 800,0000  800,0000
 14    0      0    7 28:28:7:0        sim 5000,0000 800,0000  800,0000
 15    0      0    7 28:28:7:0        sim 5000,0000 800,0000 1643,5400
 16    0      0    8 32:32:8:0        sim 3700,0000 800,0000 1668,3220
 17    0      0    9 33:33:8:0        sim 3700,0000 800,0000 1606,9969
 18    0      0   10 34:34:8:0        sim 3700,0000 800,0000 1616,8290
 19    0      0   11 35:35:8:0        sim 3700,0000 800,0000 1734,0160
 20    0      0   12 36:36:9:0        sim 3700,0000 800,0000 1356,2500
 21    0      0   13 37:37:9:0        sim 3700,0000 800,0000 1301,2190
 22    0      0   14 38:38:9:0        sim 3700,0000 800,0000 1399,7321
 23    0      0   15 39:39:9:0        sim 3700,0000 800,0000 1424,9110
 24    0      0   16 40:40:10:0       sim 3700,0000 800,0000 1413,5179
 25    0      0   17 41:41:10:0       sim 3700,0000 800,0000 1167,7300
 26    0      0   18 42:42:10:0       sim 3700,0000 800,0000 1361,3300
 27    0      0   19 43:43:10:0       sim 3700,0000 800,0000 1221,4680

In [11]:
import pandas
print(pandas.__version__)

2.1.4


In [21]:
import os
import pandas as pd

# Número máximo de CPUs a listar
max_cpu = 28

# Lista para armazenar os dados
cpu_data = []

# Loop por CPUs
for i in range(max_cpu + 1):
    cpu_path = f"/sys/devices/system/cpu/cpu{i}/cpufreq"
    
    if not os.path.isdir(cpu_path):
        continue  # pula CPUs inexistentes

    def read_file(fname):
        try:
            with open(os.path.join(cpu_path, fname), "r") as f:
                return f.read().strip()
        except FileNotFoundError:
            return "N/A"

    base_freq = read_file("base_frequency")
    cpuinfo_max = read_file("cpuinfo_max_freq")
    cpuinfo_min = read_file("cpuinfo_min_freq")
    scaling_max = read_file("scaling_max_freq")
    scaling_min = read_file("scaling_min_freq")
    driver = read_file("scaling_driver")
    governor = read_file("scaling_governor")

    cpu_data.append({
        "CPU": i,
        "base_freq(kHz)": base_freq,
        "cpu_info_max": cpuinfo_max,
        "cpu_info_min": cpuinfo_min,
        "scaling_max": scaling_max,
        "scaling_min": scaling_min,
        "governor": f"{governor}",
        "driver": f"{driver}"
    })

# Cria DataFrame
df = pd.DataFrame(cpu_data)

# Imprime o DataFrame
pd.set_option('display.width', 1000)  # Evita quebra de linha do print do df
print(df)


    CPU base_freq(kHz) cpu_info_max cpu_info_min scaling_max scaling_min     governor        driver
0     0        1300000      5000000       800000     5000000      800000  performance  intel_pstate
1     1        1300000      5000000       800000     5000000      800000  performance  intel_pstate
2     2        1300000      5000000       800000     5000000      800000  performance  intel_pstate
3     3        1300000      5000000       800000     5000000      800000  performance  intel_pstate
4     4        1300000      5000000       800000     5000000      800000  performance  intel_pstate
5     5        1300000      5000000       800000     5000000      800000  performance  intel_pstate
6     6        1300000      5000000       800000     5000000      800000  performance  intel_pstate
7     7        1300000      5000000       800000     5000000      800000  performance  intel_pstate
8     8        1300000      5200000       800000     5200000      800000  performance  intel_pstate


In [4]:
!lstopo --no-io hardware_28logical_cores.png

Exporting format `png' to file `hardware_28logical_cores.png'


### Memória

- bank:0
  - descrição: SODIMM Síncrono 5600 MHz (0,2 ns)
  - produto: AI1V56WCSV1-B1CS
  - fabricante: Fujitsu
  - ID físico: 0
  - serial: 03018662
  - slot: DIMM1
  - tamanho: 16GiB
  - largura: 64 bits
  - clock: 1305MHz (0.8ns)

In [6]:
!free -h

               total       usada       livre    compart.  buff/cache  disponível
Mem.:           15Gi       2,7Gi        11Gi       159Mi       1,9Gi        12Gi
Swap:             0B          0B          0B


In [7]:
!grep -E 'MemTotal|MemFree|SwapTotal|SwapFree' /proc/meminfo


MemTotal:       16050644 kB
MemFree:        11561280 kB
SwapTotal:             0 kB
SwapFree:              0 kB


In [10]:
import getpass, os

password = getpass.getpass("Sudo password: ")
bash_command = "lshw -class memory"

command = f'echo "{password}" | sudo -S bash -c \'{bash_command}\''
os.system(command)

Sudo password: ········


[sudo] senha para jgross: 

  *-firmware
       descrição: BIOS
       fabricante: Dell Inc.
       ID físico: 0
       versão: 1.16.1
       date: 05/14/2025
       tamanho: 1MiB
       capacidade: 48MiB
       capacidades: pci pnp upgrade shadowing cdboot bootselect edd int5printscreen int9keyboard int14serial int17printer acpi usb biosbootspecification netboot uefi
  *-cache:0
       descrição: L1 cache
       ID físico: 705
       slot: L1 Cache
       tamanho: 768KiB
       capacidade: 768KiB
       capacidades: synchronous internal write-back instruction
       configuração: level=1
  *-cache:1
       descrição: L2 cache
       ID físico: 706
       slot: L2 Cache
       tamanho: 12MiB
       capacidade: 12MiB
       capacidades: synchronous internal write-back unified
       configuração: level=2
  *-cache:2
       descrição: L3 cache
       ID físico: 707
       slot: L3 Cache
       tamanho: 33MiB
       capacidade: 33MiB
       capacidades: synchronous internal write-back unified
       configuração: le

0

### Disco

### Outros comandos

In [11]:
!numactl --hardware

available: 1 nodes (0)
node 0 cpus: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27
node 0 size: 15674 MB
node 0 free: 11424 MB
node distances:
node   0 
  0:  10 
